In [2]:
import pandas as pd
import json

# =========================================================
# 1. DOSYA YOLLARI
# =========================================================
PATH_RAW = '../data/raw/'

# =========================================================
# 2. VERİLERİ OKUMA
# =========================================================
users = pd.read_csv(f'{PATH_RAW}users_data.csv')
cards = pd.read_csv(f'{PATH_RAW}cards_data.csv')

In [3]:
# Büyük dosya olduğu için şimdilik ilk 100.000 satır
transactions = pd.read_csv(f'{PATH_RAW}transactions_data.csv')

with open(f'{PATH_RAW}mcc_codes.json', 'r') as f:
    mcc_codes = json.load(f)

with open(f'{PATH_RAW}train_fraud_labels.json', 'r') as f:
    fraud_labels = json.load(f)

print("✅ Tüm veriler başarıyla yüklendi!")

✅ Tüm veriler başarıyla yüklendi!


In [4]:
# 3. İLK KONTROLLER
# =========================================================
print("\n--- Veri Boyutları ---")
print(f"Users shape: {users.shape}")
print(f"Cards shape: {cards.shape}")
print(f"Transactions shape: {transactions.shape}")

print("\n--- Transactions sütunları ---")
print(transactions.columns.tolist())

print("\n--- Users sütunları ---")
print(users.columns.tolist())

print("\n--- Cards sütunları ---")
print(cards.columns.tolist())



--- Veri Boyutları ---
Users shape: (2000, 14)
Cards shape: (6146, 13)
Transactions shape: (13305915, 12)

--- Transactions sütunları ---
['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors']

--- Users sütunları ---
['id', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards']

--- Cards sütunları ---
['id', 'client_id', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web']


In [5]:
# 4. TRANSACTIONS TEMİZLİĞİ
# =========================================================

# Amount: '$' işaretini kaldır, float yap
transactions['amount'] = (
    transactions['amount']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

# Date: datetime yap
transactions['date'] = pd.to_datetime(transactions['date'], errors='coerce')

# Zip: string yap, .0 temizle
transactions['zip'] = (
    transactions['zip']
    .astype(str)
    .str.replace('.0', '', regex=False)
)

# Negatif tutarlar iade olabilir
transactions['is_return'] = transactions['amount'] < 0
transactions['amount'] = transactions['amount'].abs()

print("\n✅ Transactions temizliği tamamlandı!")



✅ Transactions temizliği tamamlandı!


In [6]:
# 5. ID TİPLERİNİ EŞİTLE
# =========================================================
users['id'] = users['id'].astype(str)
cards['id'] = cards['id'].astype(str)
cards['client_id'] = cards['client_id'].astype(str)

transactions['id'] = transactions['id'].astype(str)
transactions['client_id'] = transactions['client_id'].astype(str)
transactions['card_id'] = transactions['card_id'].astype(str)

print("\n--- ID tipleri ---")
print("users['id']:", users['id'].dtype)
print("cards['id']:", cards['id'].dtype)
print("cards['client_id']:", cards['client_id'].dtype)
print("transactions['id']:", transactions['id'].dtype)
print("transactions['client_id']:", transactions['client_id'].dtype)
print("transactions['card_id']:", transactions['card_id'].dtype)


--- ID tipleri ---
users['id']: object
cards['id']: object
cards['client_id']: object
transactions['id']: object
transactions['client_id']: object
transactions['card_id']: object


In [7]:
# 6. TRANSACTIONS + USERS MERGE
# =========================================================
df_combined = pd.merge(
    transactions,
    users,
    left_on='client_id',
    right_on='id',
    how='left',
    suffixes=('', '_user')
)

print("\n✅ Transactions + Users birleştirildi")
print("Shape:", df_combined.shape)


✅ Transactions + Users birleştirildi
Shape: (13305915, 27)


In [8]:
# users tablosundan gelen id sütununu kaldır
if 'id_user' in df_combined.columns:
    df_combined = df_combined.drop(columns=['id_user'])

# Bazı durumlarda merge sonrası users.id direkt 'id_y' olabilir
if 'id_y' in df_combined.columns:
    df_combined = df_combined.drop(columns=['id_y'])


In [9]:
# 7. CARDS TABLOSUNU EKLE
# transactions.card_id  <-> cards.id
# =========================================================
df_combined = pd.merge(
    df_combined,
    cards,
    left_on='card_id',
    right_on='id',
    how='left',
    suffixes=('', '_card')
)

print("\n✅ Cards tablosu eklendi")
print("Shape:", df_combined.shape)

# cards'tan gelen gereksiz tekrar id sütununu kaldır
if 'id_card' in df_combined.columns:
    df_combined = df_combined.drop(columns=['id_card'])

# Eğer merge sonrası 'id_y' oluştuysa temizle
if 'id_y' in df_combined.columns:
    df_combined = df_combined.drop(columns=['id_y'])



✅ Cards tablosu eklendi
Shape: (13305915, 39)


In [10]:
# 8. MCC CODES EKLEME
# =========================================================
# mcc_codes JSON ise dict olarak gelir:
# örnek: {"1711": "Heating, Plumbing...", ...}

mcc_df = pd.DataFrame(list(mcc_codes.items()), columns=['mcc', 'mcc_description'])
mcc_df['mcc'] = mcc_df['mcc'].astype(str)

# transactions içindeki mcc de string olsun
df_combined['mcc'] = df_combined['mcc'].astype(str)

df_combined = pd.merge(
    df_combined,
    mcc_df,
    on='mcc',
    how='left'
)

print("\n✅ MCC açıklamaları eklendi")
print("Shape:", df_combined.shape)



✅ MCC açıklamaları eklendi
Shape: (13305915, 39)


In [11]:
# 9. FRAUD LABELS DÜZELTME VE EKLEME
# =========================================================
# JSON yapısı:
# {
#   "target": {
#       "7476734": "No",
#       "7481767": "Yes",
#       ...
#   }
# }

fraud_labels = fraud_labels['target']

fraud_df = pd.DataFrame(list(fraud_labels.items()), columns=['id', 'is_fraud'])
fraud_df['id'] = fraud_df['id'].astype(str)

# Yes/No -> 1/0
fraud_df['is_fraud'] = fraud_df['is_fraud'].map({'Yes': 1, 'No': 0})

# Fraud label transaction id üzerinden eklenmeli
# Çünkü fraud JSON key = transactions.id
df_combined = pd.merge(
    df_combined,
    fraud_df,
    on='id',
    how='left'
)

# Eksik olanları 0 kabul et
df_combined['is_fraud'] = df_combined['is_fraud'].fillna(0).astype(int)

print("\n✅ Fraud label eklendi")
print("Shape:", df_combined.shape)


✅ Fraud label eklendi
Shape: (13305915, 40)


In [12]:
# 10. TARİH FEATURE'LARI
# =========================================================
df_combined['year'] = df_combined['date'].dt.year
df_combined['month'] = df_combined['date'].dt.month
df_combined['day'] = df_combined['date'].dt.day
df_combined['hour'] = df_combined['date'].dt.hour
df_combined['day_of_week'] = df_combined['date'].dt.day_name()
df_combined['is_weekend'] = df_combined['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

print("\n✅ Tarih feature'ları oluşturuldu")


✅ Tarih feature'ları oluşturuldu


In [13]:
# 11. KONTROLLER
# =========================================================
print("\n--- Son tablo bilgisi ---")
print(df_combined.shape)

print("\n--- İlk 5 satır ---")
print(df_combined.head())

print("\n--- Fraud dağılımı ---")
print(df_combined['is_fraud'].value_counts(dropna=False))

print("\n--- Fraud oranı ---")
print(df_combined['is_fraud'].mean())

print("\n--- Eksik değer kontrolü (ilk 20 sütun) ---")
print(df_combined.isnull().sum().sort_values(ascending=False).head(20))



--- Son tablo bilgisi ---
(13305915, 46)

--- İlk 5 satır ---
        id                date client_id card_id  amount           use_chip  \
0  7475327 2010-01-01 00:01:00      1556    2972   77.00  Swipe Transaction   
1  7475328 2010-01-01 00:02:00       561    4575   14.57  Swipe Transaction   
2  7475329 2010-01-01 00:02:00      1129     102   80.00  Swipe Transaction   
3  7475331 2010-01-01 00:05:00       430    2860  200.00  Swipe Transaction   
4  7475332 2010-01-01 00:06:00       848    3915   46.41  Swipe Transaction   

   merchant_id merchant_city merchant_state    zip  ... year_pin_last_changed  \
0        59935        Beulah             ND  58523  ...                  2008   
1        67570    Bettendorf             IA  52722  ...                  2015   
2        27092         Vista             CA  92084  ...                  2008   
3        27092   Crown Point             IN  46307  ...                  2006   
4        13051       Harwood             MD  20776  ...  

In [14]:
df_combined['is_fraud'].unique()

array([0, 1])

In [15]:
df_combined['is_fraud'].sum()

13332

In [17]:
print(df_combined.shape)
df_combined.head()
df_combined["is_fraud"].value_counts()


(13305915, 46)


is_fraud
0    13292583
1       13332
Name: count, dtype: int64

In [19]:
df_combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 46 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   id                     object        
 1   date                   datetime64[ns]
 2   client_id              object        
 3   card_id                object        
 4   amount                 float64       
 5   use_chip               object        
 6   merchant_id            int64         
 7   merchant_city          object        
 8   merchant_state         object        
 9   zip                    object        
 10  mcc                    object        
 11  errors                 object        
 12  is_return              bool          
 13  current_age            int64         
 14  retirement_age         int64         
 15  birth_year             int64         
 16  birth_month            int64         
 17  gender                 object        
 18  address             

In [23]:
df_combined.to_parquet("df_combined.parquet", index=False)

In [24]:
import os
print(os.getcwd())

/Users/emre/code/emrekayaa/bank-shield-ai/notebooks
